# 24 — Pull Bertelsmann Transformation Index (BTI)

**Source:** Bertelsmann Transformation Index  
**Provider:** Bertelsmann Stiftung  
**Access:** Free direct Excel download — no registration required  
**Coverage:** 137 developing and transitioning countries  
**Frequency:** Bi-annual waves (2006, 2008, 2010, 2012, 2014, 2016, 2018, 2020, 2022, 2024)  

## What this notebook does
Downloads and processes BTI governance and democracy scores. BTI provides
governance quality sub-scores more granular than WGI and less collinear with V-Dem
than you might expect — particularly useful for `regime_backsliding` where the
democratic quality of day-to-day governance matters more than structural democracy indices.

## Wave interpolation
BTI assesses the **preceding two years** (e.g., BTI 2024 covers 2021–2023).
After loading, we assign each wave to a `survey_year` (the midpoint of the
assessment period) and linearly interpolate to annual frequency, adding a
`months_since_bti` variable for the model to discount interpolated observations.

## Features pulled

| Feature | Description |
|---|---|
| `bti_democracy` | Political transformation total score (0–10) |
| `bti_market` | Economic transformation total score (0–10) |
| `bti_governance` | Governance index (0–10) |
| `bti_stateness` | Stateness sub-score (monopoly on use of force) |
| `bti_rule_of_law` | Rule of law sub-score |
| `bti_political_participation` | Political participation sub-score |
| `bti_stability` | Stability of democratic institutions |
| `bti_integration` | Socioeconomic integration / service delivery |

## Required environment variables
```
ADLS_ACCOUNT_NAME  — Azure storage account name
ADLS_CONTAINER     — Container name (default: 'data')
```

In [ ]:
import os
import io
import re
import requests
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime
from azure.identity import DefaultAzureCredential

## Configuration

In [ ]:
ADLS_ACCOUNT_NAME = os.environ["ADLS_ACCOUNT_NAME"]
ADLS_CONTAINER    = os.getenv("ADLS_CONTAINER", "data")
RUN_DATE          = datetime.utcnow().strftime("%Y%m%d")

# BTI bulk Excel download (Bertelsmann Stiftung)
BTI_URL = "https://bti-project.org/fileadmin/api/content/en/downloads/data/BTI_2006-2024.xlsx"

# Wave year → midpoint of assessment period (approximate)
BTI_WAVE_TO_SURVEY_YEAR = {
    2006: 2005, 2008: 2007, 2010: 2009, 2012: 2011, 2014: 2013,
    2016: 2015, 2018: 2017, 2020: 2019, 2022: 2021, 2024: 2023,
}

print(f"ADLS target : abfss://{ADLS_CONTAINER}@{ADLS_ACCOUNT_NAME}.dfs.core.windows.net/raw/bti/")
print(f"Run date    : {RUN_DATE}")

## ADLS helper

In [ ]:
credential = DefaultAzureCredential()
storage_options = {
    "account_name": ADLS_ACCOUNT_NAME,
    "credential": credential,
}

def adls_path(subpath: str) -> str:
    return (
        f"abfss://{ADLS_CONTAINER}@{ADLS_ACCOUNT_NAME}"
        f".dfs.core.windows.net/{subpath}"
    )

def write_parquet(df: pd.DataFrame, subpath: str) -> None:
    path = adls_path(subpath)
    df.to_parquet(path, storage_options=storage_options, index=False, engine="pyarrow")
    print(f"  Written {len(df):,} rows → {path}")

## Download and parse BTI Excel

In [ ]:
print("Downloading BTI dataset...", end=" ", flush=True)
resp = requests.get(BTI_URL, timeout=120)
resp.raise_for_status()
print(f"done ({len(resp.content) / 1e6:.1f} MB)")

xl = pd.ExcelFile(io.BytesIO(resp.content))
print(f"Sheets: {xl.sheet_names}")

## Extract and reshape

BTI ships in wide format with one column per wave per indicator.
We melt to long (country × wave × indicator) then pivot to wide (country × wave,
one column per indicator).

In [ ]:
# Read main data sheet — sheet name may vary by release year
sheet = xl.sheet_names[0]
df_raw = xl.parse(sheet, header=0)
print(f"Raw shape: {df_raw.shape}")

# The relevant BTI aggregate columns use consistent naming patterns.
# Map: display name fragments → output column names
BTI_COL_FRAGMENTS = {
    "Political Transformation":      "bti_democracy",
    "Economic Transformation":       "bti_market",
    "Governance Index":              "bti_governance",
    "Stateness":                     "bti_stateness",
    "Rule of Law":                   "bti_rule_of_law",
    "Political Participation":       "bti_political_participation",
    "Stability of Democratic":       "bti_stability",
    "Socioeconomic":                 "bti_integration",
}

# Identify country identifier columns
id_cols = [c for c in df_raw.columns if "country" in str(c).lower() or "iso" in str(c).lower()]
print(f"Identifier columns: {id_cols}")

# Melt wave columns — BTI uses multi-level headers; flatten and filter
records = []
for col in df_raw.columns:
    col_str = str(col)
    matched_feature = None
    for fragment, feat_name in BTI_COL_FRAGMENTS.items():
        if fragment.lower() in col_str.lower():
            matched_feature = feat_name
            break
    if matched_feature is None:
        continue
    # Extract wave year from column name (e.g. "BTI 2024 Stateness")
    for wave_yr in BTI_WAVE_TO_SURVEY_YEAR:
        if str(wave_yr) in col_str:
            tmp = df_raw[id_cols + [col]].copy()
            tmp.columns = id_cols + ["value"]
            tmp["wave_year"]   = wave_yr
            tmp["survey_year"] = BTI_WAVE_TO_SURVEY_YEAR[wave_yr]
            tmp["feature"]     = matched_feature
            records.append(tmp)

if not records:
    # Fallback: print columns for manual inspection
    print("WARNING: Could not auto-parse BTI columns. Column sample:")
    print(df_raw.columns[:30].tolist())
else:
    df_long = pd.concat(records, ignore_index=True)
    # Identify which id_col is the country name
    country_col = id_cols[0] if id_cols else "Country"
    df_bti_wide = (
        df_long
        .pivot_table(index=[country_col, "wave_year", "survey_year"],
                     columns="feature", values="value", aggfunc="first")
        .reset_index()
        .rename(columns={country_col: "country_name"})
    )
    df_bti_wide.columns.name = None
    print(f"Wide BTI shape: {df_bti_wide.shape}")
    print(f"Waves: {sorted(df_bti_wide['wave_year'].unique())}")
    df_bti_wide.head()

## Interpolate to annual frequency

Linear interpolation within each country between survey years; forward-fill
the most recent wave. Adds `months_since_bti` so models can weight observations
closer to an actual measurement more heavily.

In [ ]:
CURRENT_YEAR = datetime.utcnow().year
FEATURE_COLS = [c for c in df_bti_wide.columns
                if c.startswith("bti_")]

annual_frames = []
for country, grp in df_bti_wide.groupby("country_name"):
    grp = grp.sort_values("survey_year").drop_duplicates("survey_year")
    # Build annual spine from first survey_year to CURRENT_YEAR
    yr_min, yr_max = int(grp["survey_year"].min()), CURRENT_YEAR
    spine = pd.DataFrame({"year": range(yr_min, yr_max + 1)})
    spine["country_name"] = country
    # Merge sparse survey observations
    merged = spine.merge(
        grp[["survey_year"] + FEATURE_COLS].rename(columns={"survey_year": "year"}),
        on="year", how="left"
    )
    # Interpolate numeric BTI scores
    for fc in FEATURE_COLS:
        if fc in merged.columns:
            merged[fc] = merged[fc].interpolate(method="linear", limit_direction="forward")
    # Forward-fill beyond last wave
    merged[FEATURE_COLS] = merged[FEATURE_COLS].ffill()
    # Months since nearest BTI survey year
    survey_years = sorted(grp["survey_year"].astype(int).tolist())
    merged["nearest_survey_year"] = merged["year"].apply(
        lambda y: min(survey_years, key=lambda s: abs(s - y))
    )
    merged["months_since_bti"] = (merged["year"] - merged["nearest_survey_year"]).abs() * 12
    annual_frames.append(merged)

df_bti = pd.concat(annual_frames, ignore_index=True)
df_bti = df_bti.sort_values(["country_name", "year"]).reset_index(drop=True)

print(f"Annual panel shape: {df_bti.shape}")
print(f"Countries  : {df_bti['country_name'].nunique()}")
print(f"Year range : {df_bti['year'].min()}–{df_bti['year'].max()}")
df_bti.head()

## Map country names to ISO3

The downstream feature matrix joins on `iso3`. Use the project crosswalk to
translate BTI's country names; rows that can't be matched are dropped with a
warning so the parquet can be merged without surprises in `02/02`.

In [ ]:
_cw_path = Path("../../data/country_crosswalk.csv")
if not _cw_path.exists():
    _cw_path = Path("../data/country_crosswalk.csv")
if not _cw_path.exists():
    _cw_path = Path("data/country_crosswalk.csv")

df_cw = pd.read_csv(_cw_path, dtype=str)
_name_lookup = dict(zip(
    df_cw["country_name_canonical"].str.lower().str.strip(),
    df_cw["iso3"],
))
_NAME_OVERRIDES = {
    "cote d'ivoire": "CIV", "ivory coast": "CIV",
    "korea, north": "PRK", "korea, south": "KOR",
    "north korea": "PRK", "south korea": "KOR",
    "congo, dem. rep.": "COD", "dr congo": "COD",
    "democratic republic of the congo": "COD", "congo, rep.": "COG",
    "syria": "SYR", "iran": "IRN", "russia": "RUS",
    "viet nam": "VNM", "vietnam": "VNM",
    "lao pdr": "LAO", "laos": "LAO",
    "kyrgyz republic": "KGZ", "kyrgyzstan": "KGZ",
    "czech republic": "CZE", "czechia": "CZE",
    "slovak republic": "SVK", "slovakia": "SVK",
    "turkiye": "TUR", "turkey": "TUR",
    "yemen": "YEM", "venezuela": "VEN",
    "tanzania": "TZA", "bolivia": "BOL",
    "cabo verde": "CPV", "cape verde": "CPV",
    "eswatini": "SWZ", "swaziland": "SWZ",
    "north macedonia": "MKD", "macedonia": "MKD",
    "timor-leste": "TLS", "east timor": "TLS",
}

def _name_to_iso3(name: str):
    if not isinstance(name, str):
        return None
    raw_key = name.lower().strip()
    norm_key = re.sub(r"[^a-z0-9 ]", "", raw_key)
    return (
        _NAME_OVERRIDES.get(raw_key)
        or _NAME_OVERRIDES.get(norm_key)
        or _name_lookup.get(raw_key)
        or _name_lookup.get(norm_key)
    )

df_bti["iso3"] = df_bti["country_name"].apply(_name_to_iso3)
n_unmapped = df_bti["iso3"].isna().sum()
if n_unmapped:
    unmapped = df_bti.loc[df_bti["iso3"].isna(), "country_name"].unique().tolist()
    print(f"WARNING: {n_unmapped} BTI rows lacked an ISO3 match (countries: {unmapped[:15]})")
df_bti = df_bti.dropna(subset=["iso3"]).reset_index(drop=True)

# Reorder so iso3, year lead the parquet
front = ["iso3", "year", "country_name"]
df_bti = df_bti[front + [c for c in df_bti.columns if c not in front]]

print(f"After ISO3 mapping: {df_bti.shape[0]:,} rows | {df_bti['iso3'].nunique()} countries")

## Write to ADLS

In [ ]:
write_parquet(df_bti, f"raw/bti/{RUN_DATE}/bti_panel.parquet")

## Summary

In [ ]:
print("=" * 55)
print("BTI pull complete")
print("=" * 55)
print(f"  Rows (country-years) : {len(df_bti):,}")
print(f"  Countries            : {df_bti['country_name'].nunique()}")
print(f"  Year range           : {df_bti['year'].min()}–{df_bti['year'].max()}")
print(f"  BTI features         : {FEATURE_COLS}")
print()
print("ADLS path written:")
print(f"  raw/bti/{RUN_DATE}/bti_panel.parquet")